In [1]:
# --- Import libraries ---
import os
import json
import pandas as pd
import requests
from typing import List, Dict, Any, Optional
from pathlib import Path
from dotenv import load_dotenv

# Import model-specific libraries
import google.generativeai as genai
from openai import OpenAI

print("All libraries imported successfully!")

: 

In [ ]:
class LLM:
    """Unified LLM class for processing documents with different AI models."""
    
    # Class variable to track all initialized instances
    _instances = {}
    
    def __init__(self, model_type: str, model_name: str, prompt: str, **kwargs):
        """
        Initialize LLM with specified model and configuration.
        
        Args:
            model_type: Type of model ('gemini', 'claude', 'deepseek', 'openai')
            model_name: Specific model name
            prompt: Prompt template with {{DOCUMENTATION}} placeholder
            **kwargs: Additional model-specific configuration
        """
        self.model_type = model_type.lower()
        self.model_name = model_name
        self.prompt = prompt
        self.config = kwargs
        
        # Register this instance
        LLM._instances[self.model_type] = self
        
        # Initialize model-specific configurations
        self._setup_model()
        
    def _setup_model(self):
        """Setup model-specific configurations and API clients."""
        if self.model_type == 'gemini':
            self._setup_gemini()
        elif self.model_type == 'claude':
            self._setup_claude()
        elif self.model_type == 'deepseek':
            self._setup_deepseek()
        elif self.model_type == 'openai':
            self._setup_openai()
        else:
            raise ValueError(f"Unsupported model type: {self.model_type}")
    
    def _setup_gemini(self):
        """Setup Gemini API configuration."""
        api_key = os.getenv("GOOGLE_API_KEY")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY not found in environment variables")
        
        genai.configure(api_key=api_key)
        
        # Default generation config
        self.generation_config = {
            "max_output_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "top_k": self.config.get("top_k", 40)
        }
        
    def _setup_claude(self):
        """Setup Claude API configuration."""
        self.api_key = self.config.get("api_key") or os.getenv("ANTHROPIC_API_KEY")
        if not self.api_key:
            raise ValueError("ANTHROPIC_API_KEY not found")
        
        self.api_url = "https://api.anthropic.com/v1/messages"
        self.headers = {
            "x-api-key": self.api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
        
    def _setup_deepseek(self):
        """Setup DeepSeek API configuration."""
        self.api_key = self.config.get("api_key") or os.getenv("DEEPSEEK_API_KEY")
        if not self.api_key:
            raise ValueError("DEEPSEEK_API_KEY not found")
        
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
    
    def _setup_openai(self):
        """Setup OpenAI API configuration."""
        api_key = self.config.get("api_key") or os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found")
        
        self.client = OpenAI(api_key=api_key)
        
        # Default configuration
        self.openai_config = {
            "max_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "frequency_penalty": self.config.get("frequency_penalty", 0),
            "presence_penalty": self.config.get("presence_penalty", 0)
        }
    
    def query(self, content: str, max_tokens: Optional[int] = None) -> str:
        """
        Query the LLM with given content.
        
        Args:
            content: Text content to process
            max_tokens: Maximum tokens for response (overrides default)
            
        Returns:
            Model response as string
        """
        # Format prompt with content
        formatted_prompt = self.prompt.replace("{{DOCUMENTATION}}", content)
        
        if self.model_type == 'gemini':
            return self._query_gemini(formatted_prompt, max_tokens)
        elif self.model_type == 'claude':
            return self._query_claude(formatted_prompt, max_tokens)
        elif self.model_type == 'deepseek':
            return self._query_deepseek(formatted_prompt, max_tokens)
        elif self.model_type == 'openai':
            return self._query_openai(formatted_prompt, max_tokens)
    
    def _query_gemini(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Gemini model."""
        try:
            # Update max_tokens if provided
            config = self.generation_config.copy()
            if max_tokens:
                config["max_output_tokens"] = max_tokens
            
            model = genai.GenerativeModel(
                model_name=self.model_name,
                generation_config=config
            )
            
            response = model.generate_content(prompt)
            return response.text
            
        except Exception as e:
            raise Exception(f"Gemini query failed: {str(e)}")
    
    def _query_claude(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Claude model."""
        try:
            data = {
                "model": self.model_name,
                "max_tokens": max_tokens or self.config.get("max_tokens", 2000),
                "messages": [{"role": "user", "content": prompt}]
            }
            
            response = requests.post(self.api_url, headers=self.headers, json=data)
            response.raise_for_status()
            
            return response.json()["content"][0]["text"]
            
        except Exception as e:
            raise Exception(f"Claude query failed: {str(e)}")
    
    def _query_deepseek(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query DeepSeek model."""
        try:
            data = {
                "model": self.model_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens or self.config.get("max_tokens", 2000)
            }
            
            response = requests.post(self.api_url, headers=self.headers, json=data)
            response.raise_for_status()
            
            return response.json()["choices"][0]["message"]["content"]
            
        except Exception as e:
            raise Exception(f"DeepSeek query failed: {str(e)}")
    
    def _query_openai(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query OpenAI model."""
        try:
            # Update max_tokens if provided
            config = self.openai_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            raise Exception(f"OpenAI query failed: {str(e)}")
    
    def process_json_file(self, json_path: str, output_dir: str, filename: str = None, skip_empty: bool = True) -> None:
        """
        Process JSON file containing preprocessed PDF chunks.
        
        Args:
            json_path: Path to JSON file with document chunks
            output_dir: Directory to save CSV results
            filename: Base filename (without extension) for output CSV. If None, uses model_type + document_number + "_results"
            skip_empty: Whether to skip empty content chunks
        """
        try:
            # Construct output path
            if filename is None:
                json_name = Path(json_path).stem  # e.g., "27" from "27.json"
                filename = f"{self.model_type}_{json_name}_results"
            
            if not filename.endswith('.csv'):
                filename = f"{filename}.csv"
                
            output_path = os.path.join(output_dir, filename)
            
            # Check if output file already exists
            if os.path.exists(output_path):
                print(f"Output file {output_path} already exists. Skipping processing.")
                return
            
            # Ensure output directory exists
            os.makedirs(output_dir, exist_ok=True)
            
            # Load JSON data
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            print(f"Loaded JSON with {len(json_data)} chunks")
            
            # Extract document metadata
            doc_metadata = self._extract_document_metadata(json_data)
            print(f"Processing document: {doc_metadata['document_id']} - {doc_metadata['title']}")
            
            results = []
            
            # Process each chunk
            for chunk in json_data:
                doc_id = chunk.get('doc_id', '')
                chunk_num = chunk.get('chunk', 0)
                content = chunk.get('text', '')
                
                # Skip empty content if requested
                if skip_empty and not content.strip():
                    continue
                
                print(f"Processing chunk {chunk_num} of document {doc_id}...")
                
                try:
                    # Query the model
                    model_output = self.query(content)
                    
                    # Try to parse JSON output for success tracking
                    parsed_successfully = self._check_json_parsing(model_output)
                    
                    results.append({
                        "document_id": doc_id,
                        "chunk_number": chunk_num,
                        "model_type": self.model_type,
                        "model_name": self.model_name,
                        "parsed_successfully": parsed_successfully,
                        "raw_output": model_output.strip()
                    })
                    
                except Exception as e:
                    results.append({
                        "document_id": doc_id,
                        "chunk_number": chunk_num,
                        "model_type": self.model_type,
                        "model_name": self.model_name,
                        "parsed_successfully": False,
                        "raw_output": f"Error: {str(e)}"
                    })
            
            # Save results to CSV
            self._save_results_to_csv(results, output_path)
            print(f"Processing complete. Results saved to {output_path}")
            
        except Exception as e:
            print(f"Error processing JSON file: {e}")
    
    def process_directory(self, directory_path: str, output_dir: Optional[str] = None, 
                         file_pattern: str = "*.json") -> None:
        """
        Process all JSON files in a directory.
        
        Args:
            directory_path: Path to directory containing JSON files
            output_dir: Output directory (defaults to directory_path/results)
            file_pattern: File pattern to match (default: *.json)
        """
        directory_path = Path(directory_path)
        
        if output_dir is None:
            output_dir = directory_path / "results"
        else:
            output_dir = Path(output_dir)
        
        # Create output directory
        output_dir.mkdir(exist_ok=True)
        
        # Find matching files
        json_files = list(directory_path.glob(file_pattern))
        
        if not json_files:
            print(f"No files matching {file_pattern} found in {directory_path}")
            return
        
        print(f"Found {len(json_files)} files to process...")
        
        for json_file in json_files:
            # Generate filename for this model and document
            filename = f"{self.model_type}_{json_file.stem}_results"
            output_path = output_dir / f"{filename}.csv"
            
            # Check if output file already exists
            if output_path.exists():
                print(f"Output file {output_path} already exists. Skipping processing of {json_file.name}.")
                continue
                
            print(f"Processing {json_file.name}...")
            self.process_json_file(str(json_file), str(output_dir), filename)
    
    def test_connection(self) -> bool:
        """Test if the model connection is working."""
        test_content = "Hello, this is a test message."
        try:
            response = self.query(test_content)
            print(f"✓ {self.model_type.title()} connection successful!")
            print(f"Test response length: {len(response)} characters")
            return True
        except Exception as e:
            print(f"✗ {self.model_type.title()} connection failed: {str(e)}")
            return False
    
    def quick_test(self, test_input: str = None):
        """Quick test of this model with sample input."""
        if test_input is None:
            test_input = """
            Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
            IEC standards currently specify a partial safety factor of 1.35,
            but in hurricane-prone areas of the U.S., API standards require additional robustness checks
            using a 500-year return period for L2 structures.
            """
        
        print(f"--- {self.model_type.title()} Response ---")
        try:
            response = self.query(test_input)
            print(response[:500] + "..." if len(response) > 500 else response)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("\n" + "="*50 + "\n")
    
    def analyze_results(self, csv_path: str):
        """Analyze results from a processed CSV file."""
        df = pd.read_csv(csv_path)
        
        print(f"Results Analysis for {csv_path}")
        print("-" * 50)
        print(f"Total chunks processed: {len(df)}")
        print(f"Successful parses: {df['parsed_successfully'].sum()}")
        print(f"Failed parses: {(~df['parsed_successfully']).sum()}")
        print(f"Success rate: {df['parsed_successfully'].mean():.1%}")
        
        if 'model_type' in df.columns:
            print(f"Model type: {df['model_type'].iloc[0]}")
            print(f"Model name: {df['model_name'].iloc[0]}")
        
        # Show sample outputs
        print("\nSample successful outputs:")
        successful = df[df['parsed_successfully'] == True]
        if len(successful) > 0:
            for i, row in successful.head(2).iterrows():
                print(f"Chunk {row['chunk_number']}: {row['raw_output'][:200]}...")
        
        return df
    
    # Static methods for multi-model operations
    @staticmethod
    def get_working_models():
        """Get all working model instances."""
        working_models = {}
        for name, instance in LLM._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
        return working_models
    
    @staticmethod
    def test_all_connections():
        """Test all model connections."""
        print("Testing model connections...\n")
        
        working_models = {}
        for name, instance in LLM._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
            print()
        
        print(f"Working models: {list(working_models.keys())}")
        return working_models
    
    @staticmethod
    def process_with_all_models(json_path: str, output_dir: str = "results"):
        """Process a JSON file with all working models."""
        working_models = LLM.get_working_models()
        
        json_path = Path(json_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        
        print(f"Processing {json_path.name} with all working models...\n")
        
        for name, model in working_models.items():
            # Generate filename for this model
            filename = f"{name.lower()}_{json_path.stem}_results"
            output_path = output_dir / f"{filename}.csv"
            
            # Check if output file already exists
            if output_path.exists():
                print(f"Output file for {name} already exists. Skipping processing.")
                continue
                
            print(f"Processing with {name}...")
            try:
                model.process_json_file(str(json_path), str(output_dir), filename)
                print(f"✓ {name} processing complete")
            except Exception as e:
                print(f"✗ {name} processing failed: {str(e)}")
            print()
    
    @staticmethod
    def compare_models_on_file(json_path: str, models_to_compare: List[str] = None, 
                              output_dir: str = "comparison_results"):
        """
        Process the same file with multiple models for comparison.
        
        Args:
            json_path: Path to JSON file
            models_to_compare: List of model names to compare (default: all working models)
            output_dir: Directory to save comparison results
        """
        working_models = LLM.get_working_models()
        
        if models_to_compare is None:
            models_to_compare = list(working_models.keys())
        
        json_path = Path(json_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        
        print(f"Comparing models {models_to_compare} on {json_path.name}\n")
        
        results = {}
        for model_name in models_to_compare:
            if model_name in working_models:
                # Generate filename for this model
                filename = f"{model_name.lower()}_{json_path.stem}_comparison"
                output_path = output_dir / f"{filename}.csv"
                
                # Check if output file already exists
                if output_path.exists():
                    print(f"Output file for {model_name} already exists. Skipping processing.")
                    results[model_name] = str(output_path)
                    continue
                    
                print(f"Processing with {model_name}...")
                try:
                    working_models[model_name].process_json_file(str(json_path), str(output_dir), filename)
                    results[model_name] = str(output_dir / f"{filename}.csv")
                    print(f"✓ {model_name} complete")
                except Exception as e:
                    print(f"✗ {model_name} failed: {str(e)}")
                    results[model_name] = f"Error: {str(e)}"
            else:
                print(f"⚠ {model_name} not available")
            print()
        
        print("Comparison Results:")
        for model, result in results.items():
            print(f"  {model}: {result}")
        
        return results
    
    @staticmethod
    def quick_test_all_models(test_input: str = None):
        """Quick test of all working models with sample input."""
        working_models = LLM.get_working_models()
        
        if test_input is None:
            test_input = """
            Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
            IEC standards currently specify a partial safety factor of 1.35,
            but in hurricane-prone areas of the U.S., API standards require additional robustness checks
            using a 500-year return period for L2 structures.
            """
        
        print("Testing all working models with sample input...\n")
        
        for name, model in working_models.items():
            model.quick_test(test_input)
    
    # Helper methods
    def _extract_document_metadata(self, json_data: List[Dict]) -> Dict[str, Any]:
        """Extract basic document metadata from JSON data."""
        if not json_data:
            return {"document_id": "unknown", "title": "Unknown", "total_chunks": 0}
        
        first_chunk = json_data[0]
        doc_id = first_chunk.get('doc_id', 'unknown')
        
        # Try to extract title from first chunk
        text = first_chunk.get('text', '')
        title = "Unknown"
        
        # Simple heuristic for title extraction
        lines = text.strip().split('\n')
        for line in lines[:10]:
            line = line.strip()
            if line and (line.isupper() or line.startswith('**') or line.startswith('#')):
                title = line.strip('*# ')
                break
        
        return {
            "document_id": doc_id,
            "title": title,
            "total_chunks": len(json_data)
        }
    
    def _check_json_parsing(self, output: str) -> bool:
        """Check if output contains valid JSON structure."""
        try:
            if "```json" in output:
                json_str = output.split("```json")[1].split("```")[0].strip()
                json.loads(json_str)
                return True
            else:
                # Try parsing the entire output as JSON
                json.loads(output)
                return True
        except (json.JSONDecodeError, IndexError):
            return False
    
    def _save_results_to_csv(self, results: List[Dict], output_path: str) -> None:
        """Save processing results to CSV."""
        df = pd.DataFrame(results)
        df.to_csv(output_path, index=False)

print("LLM class defined successfully!")

In [ ]:
# Load environment variables
load_dotenv("project_folder/LLM/API keys/API_keys.env")

In [ ]:
# --- Import prompts ---
try:
    from prompt_gemini_V1 import prompt as gemini_prompt
    print("✓ Gemini prompt imported")
except ImportError:
    print("⚠ Gemini prompt not found, using default")
    gemini_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite import prompt as claude_prompt
    print("✓ Claude prompt imported")
except ImportError:
    print("⚠ Claude prompt not found, using default")
    claude_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_deepseek_V1 import prompt as deepseek_prompt
    print("✓ DeepSeek prompt imported")
except ImportError:
    print("⚠ DeepSeek prompt not found, using default")
    deepseek_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_gemini_V1 import prompt as openai_prompt
    print("✓ OpenAI prompt imported")
except ImportError:
    print("⚠ OpenAI prompt not found, using default")
    openai_prompt = "Analyze the following document: {{DOCUMENTATION}}"

In [ ]:
# --- Initialize LLM instances ---

# Gemini LLM
gemini = LLM(
    model_type="gemini",
    model_name="gemini-1.5-pro",
    prompt=gemini_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95,
    top_k=40
)

# Claude LLM
claude = LLM(
    model_type="claude",
    model_name="claude-3-haiku-20240307",
    prompt=claude_prompt,
    max_tokens=2000
)

# DeepSeek LLM
deepseek = LLM(
    model_type="deepseek",
    model_name="deepseek-chat",
    prompt=deepseek_prompt,
    max_tokens=2000
)

# OpenAI GPT-4.1
gpt41 = LLM(
    model_type="openai",
    model_name="gpt-4.1",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4.1-mini
gpt41_mini = LLM(
    model_type="openai",
    model_name="gpt-4.1-mini",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4.1-nano
gpt41_nano = LLM(
    model_type="openai",
    model_name="gpt-4.1-nano",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4o-mini
gpt4o_mini = LLM(
    model_type="openai",
    model_name="gpt-4o-mini",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

print("All LLM instances created successfully!")
print("Available OpenAI models: GPT-4.1, GPT-4.1-mini, GPT-4.1-nano, GPT-4o-mini")

In [ ]:
# --- Test all model connections ---
# working_models = LLM.test_all_connections()

In [ ]:
# --- Example usage for single model operations ---

# # Process with specific model
# json_file_path = "project_folder/LLM/Preprocessing/27.json"
# json_directory = "project_folder/LLM/Preprocessing"
# output_directory = "project_folder/LLM/Results"

# Single model operations
# gemini.process_json_file(json_file_path, "gemini_27_results.csv")
# gemini.process_directory(json_directory, output_directory)
# gemini.quick_test()
# gemini.analyze_results("gemini_27_results.csv")

In [ ]:
# # --- Example usage for multi-model operations ---

# # Multi-model operations (static methods)
# # LLM.process_with_all_models(json_file_path)
# # LLM.compare_models_on_file(json_file_path, ["Gemini", "Claude"])
# # LLM.quick_test_all_models()

# print("Multi-model operations examples ready!")
# print("\nExample usage:")
# print("# LLM.process_with_all_models(json_file_path)")
# print("# LLM.compare_models_on_file(json_file_path, ['Gemini', 'Claude'])")
# print("# LLM.quick_test_all_models()")

In [ ]:
# --- Advanced usage examples ---

def batch_process_with_specific_model(model_instance, directory_path: str, output_dir: str = None):
    """Helper function to process entire directory with a specific model."""
    model_instance.process_directory(directory_path, output_dir)

def compare_specific_models(json_path: str, model_list: List[str]):
    """Helper function to compare specific models."""
    return LLM.compare_models_on_file(json_path, model_list)

# Usage examples:
# batch_process_with_specific_model(gemini, json_directory, output_directory)
# results = compare_specific_models(json_file_path, ["Gemini", "DeepSeek"])

print("Advanced usage functions defined!")

### _Documents to run_: 3, 21, 26, 27, 12, 4, 16, 37, 6, 13, 17, 25

In [ ]:
# json_file_path_27 = "project_folder/LLM/Json_docs/27.json"
# json_file_path_3 = "project_folder/LLM/Json_docs/3.json"
# json_file_path_12 = "project_folder/LLM/Json_docs/12.json"
# json_file_path_4 = "project_folder/LLM/Json_docs/4.json"
# json_file_path_16 = "project_folder/LLM/Json_docs/16.json"
# json_file_path_37 = "project_folder/LLM/Json_docs/37.json"
# json_file_path_6 = "project_folder/LLM/Json_docs/6.json"
# json_file_path_13 = "project_folder/LLM/Json_docs/13.json"
# json_file_path_17 = "project_folder/LLM/Json_docs/17.json"
json_file_path_25 = "project_folder/LLM/Json_docs/25.json"
# 
json_directory = "project_folder/LLM/Json_docs"
output_directory = "project_folder/LLM/LLM_Results"

# Gemini

In [ ]:
gemini.process_json_file(json_file_path_25, output_directory, "gemini_25_results")

# Claude

In [ ]:
claude.process_json_file(json_file_path, output_directory, "claude_27_results")

# Deepseek

In [ ]:
deepseek.process_json_file(json_file_path, output_directory, "deepseek_27_results")

# OpenAI Models Testing

In [ ]:
gpt41.process_json_file(json_file_path, output_directory, "gpt41_27_results")

In [ ]:
gpt41_mini.process_json_file(json_file_path, output_directory, "gpt41_mini_27_results")

In [ ]:
gpt41_nano.process_json_file(json_file_path, output_directory, "gpt41_nano_27_results")

In [ ]:
gpt4o_mini.process_json_file(json_file_path, output_directory, "gpt4o_mini_27_results")


# Testing document 21

In [ ]:
json_file_path_21 = "project_folder/LLM/Json_docs/21.json"
json_directory = "project_folder/LLM/Json_docs"
output_directory = "project_folder/LLM/LLM_Results"

In [ ]:
gemini.process_json_file(json_file_path_21, output_directory)

In [ ]:
claude.process_json_file(json_file_path_21, output_directory)

In [ ]:
deepseek.process_json_file(json_file_path_21, output_directory)

In [ ]:
gpt4o_mini.process_json_file(json_file_path_21, output_directory)

# Testing document 26

In [ ]:
json_file_path_26 = "project_folder/LLM/Json_docs/26.json"
json_directory = "project_folder/LLM/Json_docs"
output_directory = "project_folder/LLM/LLM_Results"

In [ ]:
gemini.process_json_file(json_file_path_26, output_directory)

In [ ]:
claude.process_json_file(json_file_path_26, output_directory)

In [ ]:
deepseek.process_json_file(json_file_path_26, output_directory)

In [ ]:
gpt4o_mini.process_json_file(json_file_path_26, output_directory)